# NBA 球员下赛季 PER 预测 —— 特征工程优化 + XGBoost 调优

基于 `03_prediction_advance.ipynb` 的回归流程继续优化，目标是在**相同测试集**（`Year > 2010`）上提升 R²。

主要改动：
1. **特征工程**：追加 `PER_diff`（环比趋势）、`Career_Year`（生涯年份）、`PER_3yr_avg`（近三年 PER 均值）；
2. **模型调优**：`XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8)`；
3. **输出对比**：LinearRegression（Baseline）vs XGBRegressor（优化前）vs XGBRegressor（优化后）。

> 本 notebook 仅专注回归任务，不涉及分类、SHAP 与模型导出。

## 1. 数据准备与读取

### 1.1 导入依赖库

In [7]:
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

print(f"pandas       : {pd.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"xgboost      : {xgb.__version__}")

pandas       : 2.3.3
scikit-learn : 1.9.0
xgboost      : 2.1.4


### 1.2 读取主数据集

In [8]:
DATA_PATH = "../data/processed/master_data.csv"

df_raw = pd.read_csv(DATA_PATH)

print(f"主数据集规模: {df_raw.shape[0]} 行 x {df_raw.shape[1]} 列")
print(f"赛季覆盖范围: {df_raw['Year'].min()} - {df_raw['Year'].max()}")

主数据集规模: 20313 行 x 78 列
赛季覆盖范围: 1950 - 2017


## 2. 特征工程

在原有当季统计特征基础上，追加三个反映 **球员状态变化与生涯阶段** 的时序特征。

### 2.1 构造时序与生涯特征

- `PER_diff`：当季 PER − 上一赛季 PER，衡量上升/下滑趋势。为避免球员**断档复出**（如 2010 年后跳过 2011，2012 年复出）产生跨多个赛季的伪差值，仅当相邻两条记录的赛季年份差恰为 1 时才计算，否则置为 NaN；新秀首季不存在上一赛季，从可解释性角度将 `PER_diff` 设为 0；
- `Career_Year`：球员职业生涯第几年，用于捕捉新秀期—巅峰期—衰退期规律；
- `PER_3yr_avg`：近三年 PER 移动平均（含当前赛季），衡量基础稳定度。

In [9]:
# 先按 (Player, Year) 排序，保证组内计算严格按时间先后进行
df = df_raw.sort_values(["Player", "Year"]).reset_index(drop=True)

# 回归标签：下赛季 PER（与先前 notebook 完全一致）
df["PER_next"] = df.groupby("Player")["PER"].shift(-1)

# ---- 1) 当季 PER 相对上一赛季的变化 ----
# 仅当“上一行记录的赛季年份”与当前年份差为 1 时才计算差值；
# 若球员中间断档（如 2010 -> 2012），差值会跨多个赛季，直接置为 NaN。
prev_year = df.groupby("Player")["Year"].shift(1)
prev_PER = df.groupby("Player")["PER"].shift(1)
df["PER_diff"] = np.where(
    df["Year"] - prev_year == 1,
    df["PER"] - prev_PER,
    np.nan,
)

# ---- 2) 职业生涯第几年（首个赛季记作第 1 年）----
df["Career_Year"] = (
    df["Year"] - df.groupby("Player")["Year"].transform("min") + 1
)

# 新秀首季没有“上一赛季”，趋势差在语义上应视为 0，
# 而不是由 SimpleImputer 用全局中位数替换。
df.loc[df["Career_Year"] == 1, "PER_diff"] = 0.0

# ---- 3) 近三年 PER 移动平均（min_periods=1：前两季用已有年份计算）----
df["PER_3yr_avg"] = df.groupby("Player")["PER"].transform(
    lambda s: s.rolling(3, min_periods=1).mean()
)

print(f"新秀首季 (Career_Year==1) 样本数: {int((df['Career_Year'] == 1).sum())}")
print(f"PER_diff 处理断档后仍缺失样本数: {int(df['PER_diff'].isna().sum())} (将由 SimpleImputer 中位数填补)")
print()
print("新增特征后的缺失情况：")
print(df[["PER_diff", "Career_Year", "PER_3yr_avg"]].isna().sum().to_string())

新秀首季 (Career_Year==1) 样本数: 3922
PER_diff 处理断档后仍缺失样本数: 976 (将由 SimpleImputer 中位数填补)

新增特征后的缺失情况：
PER_diff       976
Career_Year      0
PER_3yr_avg    367


### 2.2 剔除无关字段、Pos 独热编码并扩展 `feature_cols`

In [10]:
# 与先前 notebook 保持一致的剔除字段
drop_cols = [
    "Player",
    "Tm",
    "college",
    "birth_year",
    "birth_city",
    "birth_state",
    "birth_date",
    "year_start",
    "year_end",
    "position_career",
]

df_model = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

# 原统计特征 + 本次新增的时序/生涯特征
feature_cols = [
    # ---- 原有当季统计特征 ----
    "Age",
    "G",
    "MP",
    "TS%",
    "3PAr",
    "FTr",
    "USG%",
    "PTS_per36",
    "AST%",
    "TRB%",
    "WS",
    # ---- 新增时序 / 生涯特征 ----
    "PER_diff",
    "Career_Year",
    "PER_3yr_avg",
]

# Pos -> One-Hot Encoding
pos_dummies = pd.get_dummies(df_model["Pos"], prefix="Pos").astype(int)
pos_cols = pos_dummies.columns.tolist()

df_model = pd.concat(
    [df_model.drop(columns=["Pos"]), pos_dummies],
    axis=1,
)


def build_X(data: pd.DataFrame) -> pd.DataFrame:
    """构造扩展后的回归特征矩阵（统计特征 + 时序特征 + Pos 哑变量）。"""
    return data[feature_cols + pos_cols].copy()


print(f"扩展后统计/时序特征数: {len(feature_cols)}")
print(f"Pos 哑变量数         : {len(pos_cols)}")

扩展后统计/时序特征数: 14
Pos 哑变量数         : 23


## 3. 回归任务数据与时间切分

### 3.1 构造回归样本 `X_reg` / `y_reg`

删除无法匹配到下赛季 `PER_next` 的记录（最后一季）。

In [11]:
reg_df = df_model[df_model["PER_next"].notna()].copy()

X_reg = build_X(reg_df)
y_reg = reg_df["PER_next"]
year_reg = reg_df["Year"]

print(f"回归样本数: {X_reg.shape[0]}")
print(f"X_reg 维度 : {X_reg.shape[0]} 行 x {X_reg.shape[1]} 列")

回归样本数: 16260
X_reg 维度 : 16260 行 x 37 列


### 3.2 时间维度切分（Train: `Year <= 2010` / Test: `Year > 2010`）

In [12]:
def time_split(X, y, years, train_max=2010):
    """按赛季年份切分，保证测试集对训练集而言全部位于未来。"""
    train_mask = years <= train_max
    test_mask = ~train_mask
    return (
        X.loc[train_mask],
        X.loc[test_mask],
        y.loc[train_mask],
        y.loc[test_mask],
    )


X_train_reg, X_test_reg, y_train_reg, y_test_reg = time_split(X_reg, y_reg, year_reg)
train_years = year_reg[year_reg <= 2010]
test_years = year_reg[year_reg > 2010]

print("训练集:", X_train_reg.shape, "| 赛季:", train_years.min(), "-", train_years.max())
print("测试集:", X_test_reg.shape, "| 赛季:", test_years.min(), "-", test_years.max())
print()
print("测试集新增特征缺失数量:")
print(X_test_reg[["PER_diff", "Career_Year", "PER_3yr_avg"]].isna().sum().to_string())

训练集: (13883, 37) | 赛季: 1950 - 2010
测试集: (2377, 37) | 赛季: 2011 - 2016

测试集新增特征缺失数量:
PER_diff       49
Career_Year     0
PER_3yr_avg     0


## 4. 模型训练与对比评估

### 4.1 XGBRegressor（特征工程优化后）

调优配置：降低学习率并加深树结构，同时加入行/列采样以增强泛化：

```python
XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
```

In [13]:
xgb_plus_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    )),
])

xgb_plus_pipeline.fit(X_train_reg, y_train_reg)
y_pred_reg_plus = xgb_plus_pipeline.predict(X_test_reg)

r2_plus = r2_score(y_test_reg, y_pred_reg_plus)
rmse_plus = float(np.sqrt(mean_squared_error(y_test_reg, y_pred_reg_plus)))

print("[回归] XGBRegressor (特征工程优化后)")
print(f"  R2   = {r2_plus:.4f}")
print(f"  RMSE = {rmse_plus:.4f}")

[回归] XGBRegressor (特征工程优化后)
  R2   = 0.3760
  RMSE = 4.6607


### 4.2 三方指标对比

- LinearRegression：Baseline，来自 `03_prediction_baseline.ipynb`；
- XGBRegressor（优化前）：仅 XGBoost、无新增特征，来自 `03_prediction_advance.ipynb`；
- XGBRegressor（特征工程优化后）：本次实时计算结果。

In [14]:
comparison_reg = pd.DataFrame([
    {
        "Model": "LinearRegression (Baseline)",
        "R2": 0.3441,
        "RMSE": 4.7785,
    },
    {
        "Model": "XGBRegressor (优化前)",
        "R2": 0.3494,
        "RMSE": 4.7591,
    },
    {
        "Model": "XGBRegressor (特征工程优化后)",
        "R2": r2_plus,
        "RMSE": rmse_plus,
    },
])

print("三方回归指标对比表:")
print(comparison_reg.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()

# 输出相对优化前的增量
print(f"相对 XGBRegressor(优化前): R2 提升 {r2_plus - 0.3494:+.4f}, RMSE 变化 {rmse_plus - 4.7591:+.4f}")

三方回归指标对比表:
                      Model     R2   RMSE
LinearRegression (Baseline) 0.3441 4.7785
         XGBRegressor (优化前) 0.3494 4.7591
     XGBRegressor (特征工程优化后) 0.3760 4.6607

相对 XGBRegressor(优化前): R2 提升 +0.0266, RMSE 变化 -0.0984


## 5. 阶段小结

- 通过加入 `PER_diff`、`Career_Year`、`PER_3yr_avg` 三个时序/生涯特征，并配合低学习率、加深树、行/列采样的 XGBoost 配置，在相同测试集上对 R² / RMSE 进行验证；
- 当前仅输出指标对比，未做 SHAP 特征归因，也未导出 `.pkl` 模型，后续可在确认收益后继续推进。